# Treinamento de IA

Imports e verificações mais importantes dentro do codigo:

In [1]:
import os #verificar caminhos
import urllib.request #requisição de download online
import pickle #Salvar tradutores
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Biblioteca do sklearn
from sklearn.metrics import confusion_matrix
from sklearn.preprocessing import LabelEncoder

#Para usar o fasttext, tive que importar a biblioteca do gensim
import gensim
from gensim.models import KeyedVectors

# TensorFlow e Keras (tive que seperar para evitar erro)
from tensorflow.keras.preprocessing.text import Tokenizer # type: ignore
from tensorflow.keras.preprocessing.sequence import pad_sequences # type: ignore
from tensorflow.keras.utils import to_categorical # type: ignore
from tensorflow.keras.models import Sequential # type: ignore
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional # type: ignore
from tensorflow.keras.models import load_model # type: ignore
from tensorflow.keras.callbacks import EarlyStopping # type: ignore
from tensorflow.keras.optimizers import Adam # type: ignore

#Path ou caminhos
pasta_data = '../data/'
path_dados = '../data/dataset.csv' #preciso de um dataset centralizado com todas as informações
path_models = '../models/'
path_reports = '../reports/'

#Caso alguém exclua a pasta, e estou fazendo muito isso inclusive
for path in [path_models, path_reports]:
    if not os.path.exists(path):
        os.makedirs(path)

#verificação e validação isolada apenas para o fasttext (o arquivo tem 1,2GB e o github não vai subir isso)
arquivo_fasttext = 'cc.pt.300.vec.gz'
path_fasttext = os.path.join(pasta_data, arquivo_fasttext)
url_download = 'https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.pt.300.vec.gz'


if not os.path.exists(path_fasttext):
    print("O arquivo do FastText não foi encontrado e terá que baixar (1.2GB)")
    print("Não se preocupe, a instalação sera automatica e precisa aguardar, por gentileza não feche o notebook")
    try:
        urllib.request.urlretrieve(url_download, path_fasttext)
        print("Download concluido")
    except Exception as e:
        print("Erro ao baixar o arquivo")
        print("Efetue a instalação manual: https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.pt.300.vec.gz e depois insira na pasta data o arquivo que esta zipado")

2025-11-24 03:50:05.467708: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-11-24 03:50:22.046636: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-11-24 03:50:28.057138: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


O arquivo do FastText não foi encontrado e terá que baixar (1.2GB)
Não se preocupe, a instalação sera automatica e precisa aguardar, por gentileza não feche o notebook
Download concluido


Carregar o fasttext dentro da memoria RAM:

In [2]:
#carregar o processo do fasttext para a memoria ram (nunca mais carregar isso denovo)
word_fasttext = KeyedVectors.load_word2vec_format(path_fasttext, limit=500000)

Caso haja modificações dentro do csv, apenas rode a verificação novamente abaixo:

In [3]:
# Parametros (tive que criar para mexer manualmente, sem ler o codigo completo)
vocab = 20000 #limite do vocabulario (por segurança)
maxlen = 20 #tamanho maximo das frases
epochs = 100  #as epocas
batch_size = 32
unidades_lstm = 128
embedding_dim = 300 #isso são as dimensõoes para o fasttext

# Vetorização 
df = pd.read_csv(path_dados) #vetorizar os dados dentro do csv
df = df.drop_duplicates() #remover duplicatas para o treino (ainda vou continuar reiniciando o kernel por garantia)
#remover espaços fantasmas (basicamente vazio que eu mesmo estou inserindo)
df['perguntas'] = df['perguntas'].astype(str)
df['respostas'] = df['respostas'].astype(str)
df = df.drop_duplicates() #remover duplicatas

#pegar os valores com a limpeza efetuada
perguntas = df['perguntas'].values
respostas = df['respostas'].values

#todo o processo de desenvolvimento da RNN será uma LSTM com o Keras API
#Sequências (x)
tokenizer = Tokenizer(num_words=vocab, oov_token="<OOV>") #limite de palavras e o oov_token para palavras fora do vocabulario
tokenizer.fit_on_texts(perguntas)
vocab_prop = len(tokenizer.word_index) + 1 #proporção do vocabulario

#Carregar a matriz do FastText
embedding_matrix = np.zeros((vocab_prop, embedding_dim)) #criar um matrix com o fasttext

#teste com o fasttext
acertos = 0
falhas = 0

#Verificar o vocabulário da IA com o FastTest
for word, i in tokenizer.word_index.items():
    if i >= vocab_prop: continue
    if word in word_fasttext:
        embedding_matrix[i] = word_fasttext[word]
        acertos += 1
    else:
        falhas += 1

#Preparação
x_seq = tokenizer.texts_to_sequences(perguntas) #transforma todas as palavras em sequencias numericas (coluna x)
x_processadas = pad_sequences(x_seq, maxlen=maxlen, padding='post')

#Processo de Codificação Categórica (Y)
label_encoder = LabelEncoder()
y_respostas = label_encoder.fit_transform(np.array(respostas)) #transforma as respostas em numeros, aplicado o np.array para evitar erro
y_respostas = np.array(y_respostas)  # Força conversão para NumPy array, apenas para o vscode não reclamar e aparentemente o LabelEncoder não aceita Series do pandas diretamente

#vou converter os IDs para One-Hot Encoding, trabalhar apenas com o metodo binario, isso facilita a vida da rede neural, treinamento e categorização
num_classes = len(np.unique(y_respostas)) #calcular o output unico
y_categorico = to_categorical(y_respostas, num_classes=num_classes)


#salvar os tokenizer atualizado
with open(path_models + 'tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)

with open(path_models + 'label_encoder.pkl', 'wb') as f:
    pickle.dump(label_encoder, f)

print(f"Matriz pronta! {acertos} palavras encontradas. {falhas} não encontradas (gírias/erros).")

Matriz pronta! 544 palavras encontradas. 10 não encontradas (gírias/erros).


Testes da IA:

In [4]:
#Construir o modelo
model = Sequential()

#Embedding transforma os IDs em vetores densos e é aqui que ela ira aprender o "significado" e contexto das palavras
#aplicado o fastext (um cérebro adulto)
model.add(Embedding(input_dim=vocab_prop, 
                    output_dim=300,
                    weights=[embedding_matrix], #injetando o fasttext
                    trainable=False, #congelar aprendizado
                    input_length=maxlen))
model.add(Bidirectional(LSTM(units=unidades_lstm))) #Modelo de empilhamento bidirecional
model.add(Dropout(0.3))
model.add(Dense(units=num_classes, activation='softmax')) #É a saida de decisão, pega os valores de LSTM e decide a resposta a retornar

model.compile(loss='categorical_crossentropy', # Função de perda para classificação
              optimizer='adam',                 # Otimizador padrão
              metrics=['accuracy'])             # Queremos ver a acurácia
model.summary() #mostrar arquitetura

#Treinar o Modelo
early_stopping = EarlyStopping(monitor='accuracy',
                               patience=15, 
                               restore_best_weights=True)
'''
fazer o uso do val_loss ou loss dentro do monitoramento é mais preciso para a aprendizagem da IA
val_accuracy ou a propia accuracy não é ensinar, apenas decorar as palavras com respostas 
'''

history_og = model.fit(x_processadas, 
                    y_categorico, 
                    epochs=epochs, 
                    batch_size=batch_size, 
                    validation_split=0.2,
                    callbacks=[early_stopping],
                    verbose = 1)

#tive trapacear um pouco junto com a IA, pois a aprendizagem não esta funcionando corretamente e esta proxima etapa é para isso

#aplicando o fine_tuning ou descongelamento do cérebro
model.layers[0].trainable = True

optimizer_fino = Adam(learning_rate=1e-5) #recompilar o aprendizado 

model.compile(loss='categorical_crossentropy', 
              optimizer=optimizer_fino, 
              metrics=['accuracy'])
model.summary() # Você verá que os "Trainable params" aumentaram muito

early_stopping_fino = EarlyStopping(monitor='val_loss', 
                                    patience=5, 
                                    restore_best_weights=True)

history_ft = model.fit(x_processadas, 
                    y_categorico, 
                    epochs=30, 
                    batch_size=32, 
                    validation_split=0.2, 
                    callbacks=[early_stopping_fino], 
                    verbose=1)

#Salvar o modelo
model.save(path_models + 'chatbotIA.keras') #antes estava usando o h5, mas era uma forma legada, atualizado para o proprio keras

c:\Users\playe\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │       166,500 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 166,500 (650.39 KB)

 Trainable params: 0 (0.00 B)

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 20, 300)        │       166,500 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 256)            │       439,296 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 22)             │         5,654 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 611,450 (2.33 MB)

 Trainable params: 611,450 (2.33 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/30
9/9 ━━━━━━━━━━━━━━━━━━━━ 2s 56ms/step - accuracy: 0.9823 - loss: 0.0825 - val_accuracy: 0.0000e+00 - val_loss: 12.5521
Epoch 2/30
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.9894 - loss: 0.0819 - val_accuracy: 0.0000e+00 - val_loss: 12.6041
Epoch 3/30
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.9787 - loss: 0.0903 - val_accuracy: 0.0000e+00 - val_loss: 12.6550
Epoch 4/30
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.9858 - loss: 0.0809 - val_accuracy: 0.0000e+00 - val_loss: 12.6984
Epoch 5/30
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.9787 - loss: 0.0814 - val_accuracy: 0.0000e+00 - val_loss: 12.7073
Epoch 6/30
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.9787 - loss: 0.0808 - val_accuracy: 0.0000e+00 - val_loss: 12.7182


Prints de Verificação:

In [5]:
#print de teste para ver a quantidade
print(f"Encontradas {len(perguntas)} perguntas e {len(respostas)} respostas.")

#prints de teste das perguntas
print("Exemplo de X (bruto):", perguntas[0])
print("Exemplo de X (processado):", x_processadas[0])

#prints de teste das respostas
print("Exemplo de Y (bruto):", respostas[0])
print("Exemplo de Y (Label):", y_respostas[0])

#print pós preparação de respostas
print(f"Total de classes (respostas únicas): {num_classes}")
print("Exemplo de Y (Categorical/One-Hot):", y_categorico[0])


Encontradas 353 perguntas e 353 respostas.
Exemplo de X (bruto): oi
Exemplo de X (processado): [249   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0]
Exemplo de Y (bruto): Oi! Tudo bem?
Exemplo de Y (Label): 10
Total de classes (respostas únicas): 22
Exemplo de Y (Categorical/One-Hot): [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


# Graficos

Anteriormente estava fazendo diversos show(), mas é inviavel analisar dessa forma e estou gerando png com os graficos

In [6]:
#Extração do histórico de treinamento
'''
acc = history.history['accuracy'] #acurácia de treino
val_acc = history.history['val_accuracy'] #acurácia de validação
loss = history.history['loss'] #perda de treino
val_loss = history.history['val_loss'] #perda de validação
'''
acc = history_og.history['accuracy'] + history_ft.history['accuracy']
val_acc = history_og.history['val_accuracy'] + history_ft.history['val_accuracy']
loss = history_og.history['loss'] + history_ft.history['loss']
val_loss = history_og.history['val_loss'] + history_ft.history['val_loss']

epochs = range(1, len(acc) + 1) #número de épocas treinadas


# Subplot de Acurácia e Perca
plt.figure(figsize=(16, 6)) #tamanho do grafico

#1. Grafico: Acurácia
plt.subplot(1, 2, 1)
plt.plot(epochs, acc, label='Acurácia (Treino)')
plt.plot(epochs, val_acc, label='Acurácia (Validação)')
plt.legend(loc='upper left')
plt.title('Acurácia de Treinamento vs. Validação')
plt.xlabel('Épocas')
plt.ylabel('Acurácia')
plt.grid(True, alpha=0.3)

#2. Grafico: Perca 
plt.subplot(1, 2, 2)
plt.plot(epochs, loss, label='Perda (Treino)')
plt.plot(epochs, val_loss, label='Perda (Validação)')
plt.legend(loc='upper left')
plt.title('Perda de Treinamento vs. Validação')
plt.xlabel('Épocas')
plt.ylabel('Perda')
plt.grid(True, alpha=0.3)

#para salvar os dois graficos
arquivo = path_reports + 'grafico_acuracia_perca.png'
plt.savefig(arquivo, bbox_inches='tight') #salvar o arquivo
plt.close() #fechameto para dar continuidade

#3. Grafico: Balanceamento de Classes
respostas_curtas = [t[:40] + "..." if len(t) > 40 else t for t in respostas]

plt.figure(figsize=(10, 8))
sns.countplot(y=respostas_curtas)
plt.title('Distribuição de Classes nas Respostas')
plt.xlabel('Contagem')
plt.ylabel('Respostas')

arquivo = path_reports + 'balanceamento_de_classes.png'
plt.savefig(arquivo, bbox_inches='tight') 
plt.close() 


#4. Grafico: Matriz de Confusão
#Gerar previsões
y_pred_prob = model.predict(x_processadas)
y_pred_classes = np.argmax(y_pred_prob, axis=1) # Pega a classe com maior probabilidade
y_classe_resposta = y_respostas  # 'y_respostas' são os resultados (são os IDs)

#Gerar a matriz e mapear
cm = confusion_matrix(y_classe_resposta, y_pred_classes)
nome_respostas = label_encoder.classes_.astype(str).tolist()
nomes_curtos = [t[:40] + "..." if len(t) > 40 else t for t in nome_respostas]

plt.figure(figsize=(14, 12))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=nomes_curtos, yticklabels=nomes_curtos)
plt.title('Matriz de Confusão')
plt.xlabel('Previsto pelo Modelo')
plt.ylabel('Valor Real')

arquivo = path_reports + 'matriz_confusao.png'
plt.savefig(arquivo, bbox_inches='tight') 
plt.close() 


#Distribuição do Comprimento das Frases
# Calcular o número de palavras em cada pergunta
df['comprimento_pergunta'] = df['perguntas'].apply(lambda x: len(str(x).split()))

plt.figure(figsize=(10, 5))
sns.histplot(data= df, x='comprimento_pergunta', bins=15, kde=True)
plt.title('Distribuição do Número de Palavras por Pergunta')
plt.xlabel('Número de Palavras')
plt.ylabel('Frequência')
# Linha vertical para mostrar nosso 'maxlen'
plt.axvline(x=20, color='red', linestyle='--', label=f'maxlen = {20}')
plt.legend()

arquivo = path_reports + 'distribuicao_de_frases.png'
plt.savefig(arquivo, bbox_inches='tight') 
plt.close() 

12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
